In [4]:
import torch 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader,Subset
import torch.nn as nn 

### VGG Architecture

In [ ]:
class VGG(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=transform
)

# Take only 1,000 training images
train_dataset = Subset(train_dataset, range(1000))

# Take only 1,000 test images
test_dataset = Subset(test_dataset, range(1000))

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

In [7]:
model = VGG()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for X, y in train_loader:
        # Forward
        logits = model(X)
        # Loss
        loss = criterion(logits, y)
        # Clear old gradients
        optimizer.zero_grad()
        # Backpropagation
        loss.backward()
        # Update parameters
        optimizer.step()
        
        running_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {running_loss / len(train_loader):.4f}"
    )

Epoch [1/10], Loss: 2.3063
Epoch [2/10], Loss: 2.2267
Epoch [3/10], Loss: 2.0791
Epoch [4/10], Loss: 2.0060
Epoch [5/10], Loss: 1.9316
Epoch [6/10], Loss: 1.8819
Epoch [7/10], Loss: 1.8296
Epoch [8/10], Loss: 1.7620
Epoch [9/10], Loss: 1.7228
Epoch [10/10], Loss: 1.6527


In [8]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X, y in test_loader:

        logits = model(X)

        predictions = torch.argmax(logits, dim=1)

        total += y.size(0)

        correct += (predictions == y).sum().item()

accuracy = correct / total

print("Test Accuracy:", accuracy)

Test Accuracy: 0.328
